# EE-411 — Fundamentals of Inference and Learning
## Exercise Session 7 — Kernel methods and backpropagation

**Learning objectives**
- Implement feature maps and kernel ridge regression.
- Compare SVM kernels on MNIST using cross-validation.
- Implement and check forward propagation, backpropagation, and gradient descent.

**Prerequisites**
- Ridge regression, kernels, and the chain rule.
- NumPy, matplotlib, scikit-learn, and Jupyter (`pip install numpy matplotlib scikit-learn notebook`).
- Keep `dataTP7.csv`, `mnist_subset.npz`, and `simple_mlp.png` beside this notebook; no download is needed.

Run the setup below, then complete each small `TODO` and run its check. Plotting, ridge fitting, and reporting are provided. Unfinished code raises `NotImplementedError`. Parts 1–3 can be started independently after setup: an unfinished kernel task need not delay the neural-network work. Within the neural network, reuse each checked computation in the next step. The three quick questions require only one sentence each and do not gate later work. Part 4 is optional and ready to run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC

plt.rcParams.update({"figure.figsize": (7, 4), "font.size": 11})

## Part 1 — Feature maps and kernel ridge regression
### Exercise 1 — Polynomial features
Load the synthetic data and build a feature matrix with one row per sample.

In [ ]:
data = np.genfromtxt("dataTP7.csv", delimiter=",", names=True)
x_raw, y_reg = data["X"], data["Y"]
# The input domain is [0, 2π]. Map it to [-1, 1] before taking high powers.
x_reg = x_raw / np.pi - 1
x_train, x_val, y_train_reg, y_val_reg = train_test_split(
    x_reg, y_reg, train_size=0.5, random_state=1926)
plt.scatter(x_reg, y_reg, label="Observations")
plt.xlabel("Scaled input x")
plt.ylabel("Target y")
plt.title("Synthetic regression data")
plt.legend()
plt.show()

Use $\phi^{(p)}(x)=(1,x,\ldots,x^p)^\top$, including the constant feature. Inputs are scaled to $[-1,1]$ to avoid enormous powers; high-degree monomials can still be ill-conditioned.

The provided ridge solver minimizes $\frac{1}{2n}\|\Phi w-y\|_2^2+\lambda\|w\|_2^2$. It penalizes all coefficients, including the constant feature.

In [ ]:
def fit_ridge(Phi, y, lambda_):
    system = Phi.T @ Phi + 2 * len(y) * lambda_ * np.eye(Phi.shape[1])
    return np.linalg.solve(system, Phi.T @ y)


def rmse(y, prediction):
    return np.sqrt(np.mean((y - prediction) ** 2))


def polynomial_features(x, degree):
    """Return an (n_samples, degree + 1) feature matrix."""
    # TODO 1: construct powers 0 through degree for each input.
    # YOUR CODE HERE
    raise NotImplementedError("Complete exercise 1")
    # END TODO 1

np.testing.assert_allclose(
    polynomial_features(np.array([-2., 0., 1.]), 2),
    [[1., -2., 4.], [1., 0., 0.], [1., 1., 1.]])
print("Polynomial-feature check passed.")

In [ ]:
degrees = [5, 20, 160]
lambda_ = 1e-5
x_grid = np.linspace(-1.0, 1.0, 300)
fig, axes = plt.subplots(len(degrees), 1, figsize=(7, 8), layout="constrained")
for ax, degree in zip(axes, degrees):
    Phi = polynomial_features(x_reg, degree)
    weights = fit_ridge(Phi, y_reg, lambda_)
    ax.scatter(x_reg, y_reg, s=12, label="Data")
    ax.plot(x_grid, polynomial_features(x_grid, degree) @ weights, label="Fit")
    ax.set(title=f"Degree {degree}; training RMSE = {rmse(y_reg, Phi @ weights):.3f}",
           xlabel="Scaled input x", ylabel="Prediction / target")
    ax.legend()
plt.show()

### Exercise 2 — Gaussian features and regularization
Compare both feature maps using the same training/validation split. Validation error is used to compare hyperparameters.

For $k_\gamma(x,z)=\exp(-(x-z)^2/\gamma)$, $\gamma>0$, the Gaussian feature expansion is
$$\psi_j(x)=e^{-x^2/\gamma}\sqrt{\frac{(2/\gamma)^j}{j!}}x^j,\qquad j=0,1,\ldots.$$
Truncate after degree $p$. Complete the recurrence from column $j-1$ to column $j$; cancelling adjacent factorials avoids computing large factorials.

In [ ]:
def gaussian_features(x, degree, gamma=1.0):
    """Return the first degree + 1 Gaussian features for 1D inputs."""
    x = np.asarray(x)
    Phi = np.empty((len(x), degree + 1))
    Phi[:, 0] = np.exp(-x**2 / gamma)
    for j in range(1, degree + 1):
        # TODO 2: compute the next column from the previous column.
        # YOUR CODE HERE
        raise NotImplementedError("Complete exercise 2")
        # END TODO 2
    return Phi

np.testing.assert_allclose(gaussian_features(np.array([0.]), 3), [[1., 0., 0., 0.]])
# For enough features, the squared row norm approaches k(x, x) = 1.
for gamma in [0.5, 1.0, 2.0]:
    Phi_check = gaussian_features(np.array([-0.8, 0.3]), 30, gamma)
    np.testing.assert_allclose(np.sum(Phi_check**2, axis=1), 1., atol=1e-10)
print("Gaussian-feature checks passed.")

In [ ]:
lambdas = np.logspace(-6, 1, 40)


def regularization_curves(feature_map, degree):
    Phi_train = feature_map(x_train, degree)
    Phi_val = feature_map(x_val, degree)
    train_errors, val_errors = [], []
    for penalty in lambdas:
        weights = fit_ridge(Phi_train, y_train_reg, penalty)
        train_errors.append(rmse(y_train_reg, Phi_train @ weights))
        val_errors.append(rmse(y_val_reg, Phi_val @ weights))
    return train_errors, val_errors

fig, axes = plt.subplots(len(degrees), 2, figsize=(12, 9), layout="constrained")
for row, degree in enumerate(degrees):
    for col, (name, feature_map) in enumerate([
        ("Monomials", polynomial_features), ("Gaussian features (γ=1)", gaussian_features)
    ]):
        train_errors, val_errors = regularization_curves(feature_map, degree)
        ax = axes[row, col]
        ax.semilogx(lambdas, train_errors, label="Training")
        ax.semilogx(lambdas, val_errors, label="Validation")
        ax.set(title=f"{name}, degree {degree}", xlabel="λ", ylabel="RMSE")
        ax.legend()
plt.show()

**Quick question 1:** Can the smallest training RMSE alone identify the model that generalizes best?

<!-- ANSWER 1 -->
_One sentence is enough._
<!-- END ANSWER 1 -->

### Exercise 3 — Compute kernels directly
The direct kernel calculation below only needs the data split and imports; it does not depend on either feature-map implementation.

For sample arrays $a$ and $b$, return a matrix with entry $K_{ij}=k_\gamma(a_i,b_j)$. In the dual formulation, solve
$$(K_{\mathrm{train},\mathrm{train}}+2n\lambda I)\alpha=y_{\mathrm{train}},\qquad
\hat y_{\mathrm{val}}=K_{\mathrm{val},\mathrm{train}}\alpha.$$
For a finite feature map, $K=\Phi\Phi^\top$. A provided comparison follows the two checks.

In [ ]:
def rbf_kernel_1d(a, b, gamma=1.0):
    """Return a kernel matrix of shape (len(a), len(b))."""
    # TODO 3: compute all pairwise squared distances and the Gaussian kernel.
    # YOUR CODE HERE
    raise NotImplementedError("Complete exercise 3")
    # END TODO 3

K_train = rbf_kernel_1d(x_train, x_train)
K_val = rbf_kernel_1d(x_val, x_train)
assert K_val.shape == (len(x_val), len(x_train))
np.testing.assert_allclose(K_train, K_train.T)
np.testing.assert_allclose(np.diag(K_train), 1)
# Check unequal input lengths, signs, and the width parameter.
np.testing.assert_allclose(
    rbf_kernel_1d(np.array([-1., 0.]), np.array([0., 1., 2.]), gamma=2),
    np.exp(-np.array([[1., 4., 9.], [0., 1., 4.]]) / 2))
print("RBF-kernel checks passed.")

In [ ]:
def kernel_predict(K_train, K_query, y_train, lambda_):
    # TODO 4: solve for the dual coefficients and predict at the query inputs.
    # YOUR CODE HERE
    raise NotImplementedError("Complete exercise 4")
    # END TODO 4

# A small independent check; it does not use either feature-map implementation.
np.testing.assert_allclose(
    kernel_predict(np.eye(2), np.array([[1., 2.]]), np.array([2., -1.]), 0.25), [0.])
np.testing.assert_allclose(
    kernel_predict(np.eye(2), np.eye(2), np.array([2., -1.]), 0.25), [1., -0.5])
print("Dual-prediction checks passed.")

Run this comparison once the Gaussian features and dual predictor pass their checks. If either is unfinished, you can move directly to Part 2 or Part 3.

In [ ]:
penalty = 1e-3
exact_prediction = kernel_predict(K_train, K_val, y_train_reg, penalty)
for degree in [2, 5, 12, 20]:
    Phi_train = gaussian_features(x_train, degree)
    Phi_val = gaussian_features(x_val, degree)
    finite_prediction = kernel_predict(
        Phi_train @ Phi_train.T, Phi_val @ Phi_train.T, y_train_reg, penalty)
    primal_prediction = Phi_val @ fit_ridge(Phi_train, y_train_reg, penalty)
    np.testing.assert_allclose(finite_prediction, primal_prediction, atol=1e-8)
    print(f"Degree {degree:2d}: max kernel error = "
          f"{np.max(np.abs(Phi_train @ Phi_train.T - K_train)):.2e}; "
          f"prediction gap = {np.max(np.abs(finite_prediction - exact_prediction)):.2e}")

**Quick question 2:** As the truncation degree increases in this experiment, does the prediction gap to the exact Gaussian kernel grow or shrink?

<!-- ANSWER 2 -->
_One sentence is enough._
<!-- END ANSWER 2 -->

## Part 2 — Kernel classification with scikit-learn
### Exercise 4 — Select an SVM kernel
Compare `linear`, `poly`, `rbf`, and `sigmoid` support vector classifiers. Fit the provided grid using cross-validation on training data; inspect the report, then move on. The test set is used only for the selected model.

`mnist_subset.npz` contains the first 10,500 images and labels from the standard MNIST training split (28 × 28 pixels, uint8; 1.65 MiB compressed). Source: the TensorFlow/Keras MNIST archive (`storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz`). All samples are used in the 9,000/1,500 split below. Keeping this small subset makes the exercise runnable offline.

In [ ]:
with np.load("mnist_subset.npz", allow_pickle=False) as mnist:
    images, labels = mnist["images"], mnist["labels"]
plt.imshow(images[6542], cmap="gray")
plt.title(f"MNIST example — label {labels[6542]}")
plt.axis("off")
plt.show()

# Fixed pixel scaling requires no estimates from validation or test samples.
X_digits = images.reshape(len(images), -1).astype(np.float64) / 255.0
X_train_digits, X_test_digits, y_train_digits, y_test_digits = train_test_split(
    X_digits, labels, train_size=9000, test_size=1500,
    stratify=labels, random_state=0)

# A compact grid keeps the comparison practical; full execution may take several minutes.
parameter_grid = [
    {"kernel": ["linear"], "C": [0.1, 1.0]},
    {"kernel": ["poly"], "C": [1.0], "degree": [2, 3],
     "gamma": ["scale"], "coef0": [1.0]},
    {"kernel": ["rbf"], "C": [0.1, 1.0], "gamma": ["scale"]},
    {"kernel": ["sigmoid"], "C": [1.0], "gamma": [0.001, 0.01], "coef0": [0.0]},
]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

In [ ]:
# TODO 5: fit GridSearchCV with SVC(), parameter_grid, cv, accuracy scoring, and n_jobs=1; name it search.
# YOUR CODE HERE
raise NotImplementedError("Complete exercise 5")
# END TODO 5

assert search.n_splits_ == 3
assert len(search.cv_results_["params"]) == sum(
    len(grid["C"]) * len(grid.get("degree", [None])) * len(grid.get("gamma", [None]))
    for grid in parameter_grid)
print("Cross-validation finished; inspect the report below.")

In [ ]:
results = search.cv_results_
for kernel in ["linear", "poly", "rbf", "sigmoid"]:
    candidates = [i for i, params in enumerate(results["params"]) if params["kernel"] == kernel]
    best = max(candidates, key=lambda i: results["mean_test_score"][i])
    print(f"{kernel:7s}: CV accuracy = {results['mean_test_score'][best]:.3f}; "
          f"parameters = {results['params'][best]}")
print("Selected parameters:", search.best_params_)
print(f"Selected model's test accuracy: {search.score(X_test_digits, y_test_digits):.3f}")

Selection is limited to this grid: it does not establish a universally best kernel. The sigmoid kernel is not positive semidefinite for all parameter choices.

## Part 3 — Forward and backward propagation
### Exercise 5 — Implement the forward pass
This part only needs the imports in the setup cell. We use $D=4$ inputs, $K=5$ hidden units, and sigmoid activations at both layers.

![Single-hidden-layer network](simple_mlp.png)

For this implementation we **omit biases**. With $W_1\in\mathbb R^{D\times K}$ and $w_2\in\mathbb R^K$,
$$z_1=W_1^\top x,\qquad h=\sigma(z_1),\qquad z_2=w_2^\top h,\qquad\hat y=\sigma(z_2).$$
Equivalently, $(z_1)_j=\sum_i(W_1)_{ij}x_i$. The sigmoid is $\sigma(t)=1/(1+e^{-t})$. Its numerically stable implementation is provided. The activation derivative is also provided. Complete the forward pass and save its intermediate values for backpropagation.

In [ ]:
def sigmoid(t):
    return np.exp(-np.logaddexp(0.0, -np.asarray(t)))


def grad_sigmoid(t):
    activation = sigmoid(t)
    return activation * (1 - activation)


def simple_feed_forward(x, parameters):
    """Return a scalar prediction and a cache for one input vector."""
    # TODO 6: compute z1, h, z2 and prediction; check the matrix shapes.
    # YOUR CODE HERE
    raise NotImplementedError("Complete exercise 6")
    # END TODO 6
    return prediction, {"x": x, "z1": z1, "h": h, "z2": z2}

x_nn = np.array([0.01, 0.02, 0.03, 0.04])
y_nn = 1.0
parameters = {"w1": np.ones((4, 5)), "w2": np.ones(5)}
prediction, cache = simple_feed_forward(x_nn, parameters)
assert np.ndim(prediction) == 0
assert cache["h"].shape == (5,)
np.testing.assert_allclose(prediction, 0.93244675427215695, rtol=0, atol=1e-12)
# Nonuniform parameters catch shape and summation mistakes hidden by all-ones weights.
check_parameters = {"w1": np.array([[1., -1., 0.], [0.5, 2., -0.5]]),
                    "w2": np.array([0.3, -0.2, 0.7])}
check_prediction, check_cache = simple_feed_forward(np.array([0.2, -0.4]), check_parameters)
np.testing.assert_allclose(check_cache["z1"], [0., -1., 0.2], atol=1e-12)
assert check_cache["h"].shape == (3,)
assert 0 < check_prediction < 1
print("Forward-pass checks passed.")

### Exercise 6 — Implement backpropagation
For one sample, $\mathcal L=\tfrac12(\hat y-y)^2$. Apply the chain rule in two small steps:
1. Compute $\delta_2=\partial\mathcal L/\partial z_2$ (scalar), then $\delta_1=\partial\mathcal L/\partial z_1$ (length $K$), using the current output weights.
2. Combine these error signals with the cached layer inputs to form gradients of shapes $(D,K)$ and $(K,)$. Use an outer product for the first layer.

No written derivation is needed. Return gradients without changing parameters; the numerical check below tests your implementation.

In [ ]:
def error_signals(y, prediction, cache, parameters):
    # TODO 7: propagate the output error to delta2 (scalar) and delta1 (length K).
    # YOUR CODE HERE
    raise NotImplementedError("Complete exercise 7")
    # END TODO 7
    return delta2, delta1

# Check the intermediate quantities before building weight gradients.
check_parameters = {"w1": np.zeros((2, 3)), "w2": np.array([1., -2., 1.])}
check_prediction, check_cache = simple_feed_forward(np.array([0.2, -0.4]), check_parameters)
delta2, delta1 = error_signals(1., check_prediction, check_cache, check_parameters)
np.testing.assert_allclose(delta2, -0.125)
np.testing.assert_allclose(delta1, [-0.03125, 0.0625, -0.03125])
print("Error-signal checks passed.")

In [ ]:
def simple_backpropagation(y, x, parameters):
    prediction, cache = simple_feed_forward(x, parameters)
    delta2, delta1 = error_signals(y, prediction, cache, parameters)
    # TODO 8: build grad_w1 and grad_w2 using the cached inputs and error signals.
    # YOUR CODE HERE
    raise NotImplementedError("Complete exercise 8")
    # END TODO 8
    return {"w1": grad_w1, "w2": grad_w2}


def sample_loss(y, x, parameters):
    prediction, _ = simple_feed_forward(x, parameters)
    return 0.5 * (prediction - y) ** 2


def finite_difference_gradients(y, x, parameters, epsilon=1e-5):
    """Independent numerical check; do not use this to train the network."""
    numerical = {}
    for name, values in parameters.items():
        numerical[name] = np.zeros_like(values)
        for index in np.ndindex(values.shape):
            plus = {key: value.copy() for key, value in parameters.items()}
            minus = {key: value.copy() for key, value in parameters.items()}
            plus[name][index] += epsilon
            minus[name][index] -= epsilon
            numerical[name][index] = (
                sample_loss(y, x, plus) - sample_loss(y, x, minus)
            ) / (2 * epsilon)
    return numerical

# Asymmetric weights and several shapes catch errors that all-ones weights can hide.
rng = np.random.default_rng(7)
for D, K in [(4, 5), (3, 2), (2, 4)]:
    check_parameters = {"w1": rng.normal(0, 0.3, (D, K)), "w2": rng.normal(0, 0.3, K)}
    check_x = rng.normal(size=D)
    for check_y in [0.0, 1.0]:
        analytical = simple_backpropagation(check_y, check_x, check_parameters)
        numerical = finite_difference_gradients(check_y, check_x, check_parameters)
        for name in check_parameters:
            assert analytical[name].shape == check_parameters[name].shape
            np.testing.assert_allclose(analytical[name], numerical[name], rtol=1e-5, atol=1e-8)
print("Finite-difference gradient checks passed.")

### Exercise 7 — Apply gradient descent
Use the checked gradients to fit one sample. Compute both gradients before changing any parameter. The plot checks optimization, not generalization.

In [ ]:
def gradient_step(parameters, gradients, learning_rate):
    """Return updated arrays without modifying the input dictionary."""
    # TODO 9: apply one gradient-descent step to every parameter.
    # YOUR CODE HERE
    raise NotImplementedError("Complete exercise 9")
    # END TODO 9

before = {name: value.copy() for name, value in parameters.items()}
gradients = simple_backpropagation(y_nn, x_nn, parameters)
after = gradient_step(parameters, gradients, 0.1)
assert sample_loss(y_nn, x_nn, after) < sample_loss(y_nn, x_nn, before)
for name in parameters:
    np.testing.assert_array_equal(parameters[name], before[name])
print("Gradient-step checks passed.")

fig, ax = plt.subplots()
for learning_rate in [0.1, 1.0, 10.0]:
    current = {name: value.copy() for name, value in parameters.items()}
    losses = [sample_loss(y_nn, x_nn, current)]
    for step in range(100):
        gradients = simple_backpropagation(y_nn, x_nn, current)
        current = gradient_step(current, gradients, learning_rate)
        losses.append(sample_loss(y_nn, x_nn, current))
    ax.semilogy(losses, label=f"η = {learning_rate}")
ax.set(xlabel="Gradient steps", ylabel="Squared-error loss", title="Fitting one sample")
ax.legend()
plt.show()

**Quick question 3:** The largest tested learning rate decreases the loss fastest here. Does that make a larger learning rate always better?

<!-- ANSWER 3 -->
_One sentence is enough._
<!-- END ANSWER 3 -->

## Part 4 — Optional: the effect of regularization
This experiment only needs NumPy and matplotlib. Run it as provided; optionally change the eigenvalues to explore the curves.

Near a local minimizer $w^\star$ of the training loss, assume
$$\mathcal L(w)\approx\mathcal L(w^\star)+\tfrac12(w-w^\star)^\top H(w-w^\star),\qquad H=Q\Lambda Q^\top\succeq0.$$
Adding $\tfrac\mu2\|w\|^2$, $\mu>0$, gives the quadratic approximation's minimizer
$$\widetilde w=Q(\Lambda+\mu I)^{-1}\Lambda Q^\top w^\star.$$
Each eigendirection retains a fraction $\lambda_i/(\lambda_i+\mu)$: small eigenvalues relative to $\mu$ shrink more, and zero eigenvalues retain nothing. This local approximation can fail if the regularized optimum moves far from $w^\star$.

In [ ]:
eigenvalues = np.array([0.0, 0.01, 1.0, 100.0])
penalties = np.logspace(-4, 4, 200)
shrinkage = eigenvalues[None, :] / (eigenvalues[None, :] + penalties[:, None])
for i, eigenvalue in enumerate(eigenvalues):
    plt.semilogx(penalties, shrinkage[:, i], label=f"Eigenvalue = {eigenvalue:g}")
plt.xlabel("Regularization strength μ")
plt.ylabel("Fraction retained in an eigendirection")
plt.title("Regularization in the quadratic approximation")
plt.legend()
plt.show()